# Notebook 02 — Interpretable Concern Signals

**Project:** Detecting Service Gaps and Bias Signals in Hospital Reviews  

---

## Goals
1. Apply lexicon-based concern flags to every review
2. Visualise concern rates overall and by sentiment
3. Demonstrate explainability: show which keywords triggered each flag
4. Run mismatch analysis (positive label + negative language)
5. Show concern rates by hospital (the key operational insight)

## 0 — Setup

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams['figure.dpi'] = 120

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

## 1 — Load Processed Data

In [ ]:
processed_path = ROOT / 'data' / 'processed' / 'reviews_clean.csv'

if processed_path.exists():
    df = pd.read_csv(processed_path)
    print(f'Loaded processed data: {df.shape}')
else:
    # Fallback: load raw and standardise
    df = pd.read_csv(ROOT / 'data' / 'hospital.csv')
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Feedback':        'review_text',
        'Sentiment Label': 'sentiment_label',
        'Ratings':         'rating',
    })
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df.drop_duplicates().dropna(subset=['review_text'])
    df['sentiment_label'] = pd.to_numeric(df['sentiment_label'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['sentiment_label'])
    df['sentiment_label'] = df['sentiment_label'].astype(int)
    df['sentiment'] = df['sentiment_label'].map({1: 'positive', 0: 'negative'})
    df = df.reset_index(drop=True)
    print(f'Loaded from raw: {df.shape}')

df.head(3)

## 2 — The Concern Lexicon

Each review is scanned for keywords in four manually curated categories.  
Every flag is **fully traceable** — you can always point to the exact word that triggered it.

In [ ]:
# Lexicon definition (mirrors src/config.py)
CONCERN_LEXICON = {
    'mistreatment': [
        'rude', 'rudely', 'ignored', 'dismissive', 'disrespectful',
        'insulted', 'humiliated', 'yelled', 'arrogant', 'negligent',
        'misbehaved', 'unprofessional', 'attitude', 'shouted', 'refused',
    ],
    'access_barriers': [
        'wheelchair', 'ramp', 'accessible', 'accessibility', 'lift',
        'elevator', 'stairs', 'disabled', 'handicap', 'parking',
        'entrance', 'navigation', 'blind', 'mobility',
    ],
    'delays_wait': [
        'waiting', 'wait', 'waited', 'hours', 'queue', 'queued',
        'delayed', 'delay', 'slow', 'late', 'forever', 'long time',
        'appointment', 'scheduled', 'overdue',
    ],
    'safety_hygiene': [
        'dirty', 'unclean', 'unhygienic', 'hygiene', 'infection',
        'infested', 'cockroach', 'rats', 'smell', 'smelly', 'stench',
        'contaminated', 'unsanitary',
    ],
}

# Print nicely for demo
print('Concern Lexicon Categories:\n')
for cat, kws in CONCERN_LEXICON.items():
    print(f'  {cat:<20}: {kws[:5]}  ... ({len(kws)} total keywords)')

## 3 — Apply Concern Flags

In [ ]:
def flag_concern(text, keywords):
    """Return 1 if any keyword appears in text (case-insensitive)."""
    t = str(text).lower()
    return int(any(kw in t for kw in keywords))

def get_matched_keywords(text, keywords):
    """Return list of matched keywords for explainability."""
    t = str(text).lower()
    return [kw for kw in keywords if kw in t]

# Apply flags
for cat, kws in CONCERN_LEXICON.items():
    df[f'flag_{cat}'] = df['review_text'].apply(lambda x: flag_concern(x, kws))

flag_cols = [f'flag_{c}' for c in CONCERN_LEXICON]
df['flag_any_concern'] = (df[flag_cols].sum(axis=1) > 0).astype(int)

total_flagged = df['flag_any_concern'].sum()
print(f'Reviews with ≥1 concern flag: {total_flagged:,} / {len(df):,}  ({total_flagged/len(df)*100:.1f}%)')
df[flag_cols + ['flag_any_concern']].sum().to_frame('flagged_count')

## 4 — Concern Rate by Category + Sentiment

In [ ]:
categories = list(CONCERN_LEXICON.keys())

concern_by_sentiment = df.groupby('sentiment')[flag_cols].mean() * 100
concern_by_sentiment.columns = categories

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width/2, concern_by_sentiment.loc['positive'] if 'positive' in concern_by_sentiment.index else [0]*len(categories),
               width, label='Positive Reviews', color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, concern_by_sentiment.loc['negative'] if 'negative' in concern_by_sentiment.index else [0]*len(categories),
               width, label='Negative Reviews', color='#e74c3c', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in categories], fontsize=10)
ax.set_ylabel('Concern Rate (%)')
ax.set_title('Concern Rate by Category × Sentiment', fontsize=13, fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'concern_drivers_negative.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/concern_drivers_negative.png')

## 5 — Explainability Demo: Flagged Examples

This is the key interview demo section — show that every flag points to specific words.

In [ ]:
print('=== EXPLAINABILITY DEMO ===\n')
print('Showing 5 randomly sampled flagged reviews with their trigger keywords:\n')

flagged_df = df[df['flag_any_concern'] == 1].sample(min(5, df['flag_any_concern'].sum()), random_state=42)

for i, (_, row) in enumerate(flagged_df.iterrows(), 1):
    print(f'[{i}] Sentiment: {row["sentiment"]} | Concern flags: {[c for c in flag_cols if row[c]==1]}')
    print(f'    Review: "{str(row["review_text"])[:200]}"')
    for cat, kws in CONCERN_LEXICON.items():
        matched = get_matched_keywords(row['review_text'], kws)
        if matched:
            print(f'    ✓ {cat}: triggered by → {matched}')
    print()

## 6 — Mismatch Analysis

> **Type A**: Positive label + concern flags → reviewer may be giving benefit of the doubt despite a negative experience  
> **Type B**: Negative label + no concern flags → negative experience driven by factors outside our lexicon

In [ ]:
type_a = df[(df['sentiment_label'] == 1) & (df['flag_any_concern'] == 1)].copy()
type_a['mismatch_type'] = 'A: positive_label + concern_flags'

type_b = df[(df['sentiment_label'] == 0) & (df['flag_any_concern'] == 0)].copy()
type_b['mismatch_type'] = 'B: negative_label + no_flags'

total = len(df)
print(f'Mismatch Analysis:')
print(f'  Type A (hidden dissatisfaction):  {len(type_a):,} reviews ({len(type_a)/total*100:.1f}%)')
print(f'  Type B (unexplained negatives):   {len(type_b):,} reviews ({len(type_b)/total*100:.1f}%)')

# Plot
labels = ['Type A\n(positive + flags)', 'Type B\n(negative + no flags)', 'Consistent\n(no mismatch)']
counts = [len(type_a), len(type_b), total - len(type_a) - len(type_b)]
colors = ['#f39c12', '#8e44ad', '#bdc3c7']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, counts, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val}\n({val/total*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Mismatch Analysis: Sentiment Label vs Concern Flags', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Reviews')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mismatch_analysis.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/mismatch_analysis.png')

In [ ]:
# Show sample Type A mismatches
print('=== SAMPLE TYPE A MISMATCHES (positive label + concern language) ===\n')
for _, row in type_a.sample(min(3, len(type_a)), random_state=1).iterrows():
    print(f'Review: "{str(row["review_text"])[:250]}"')
    for cat, kws in CONCERN_LEXICON.items():
        matched = get_matched_keywords(row['review_text'], kws)
        if matched:
            print(f'  → {cat}: {matched}')
    print()

## 7 — Save Flagged Dataset

In [ ]:
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / 'reviews_flagged.csv', index=False)
print(f'Saved {len(df):,} rows → data/processed/reviews_flagged.csv')

print('\nFlag column summary:')
print(df[flag_cols + ['flag_any_concern']].mean().mul(100).round(1).rename('Concern Rate (%)'))

---
## Summary

- Applied **4 concern signal categories** (mistreatment, access barriers, delays/wait, safety/hygiene)
- Every flag is **traceable to specific words** — no black box
- Identified **mismatch cases** most valuable for operational quality assurance

**Next:** `03_topics_and_drivers.ipynb` — LDA topic modelling on negative reviews